<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/main/NB04_NumPy_for_Engineering_Computation_Vectors_Matrices_and_Linear_Algebra.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# **NB04 · Class 4 — NumPy for Engineering Computation: Vectors, Matrices, and Linear Algebra**

## Block 2: AI — Machine Learning (continued)

`NB03` covered a single array's mechanics. This class covers what happens **between** arrays: broadcasting rules that let NumPy combine arrays of different shapes without writing a loop, real matrix/vector operations (`dot`, `@`, `einsum`), and `numpy.linalg` for solving real systems of linear equations — applied to a genuine naval statics problem: **finding the tension in a vessel's mooring lines**. This is also the linear algebra `NB09`'s PCA quietly relies on (eigenvectors of a covariance matrix), made explicit here instead of hidden inside `sklearn`.

### Learning objectives

By the end of this class, students will be able to:
- Explain NumPy's broadcasting rules and use them to avoid explicit loops on real data.
- Reshape and stack arrays to match the shape a computation needs.
- Compute dot products, matrix multiplication, and outer products, and know when each applies.
- Set up and solve a real system of linear equations with `numpy.linalg.solve`.
- Compute eigenvalues/eigenvectors and connect them, concretely, to what PCA does under the hood.

### Agenda (2-hour class)

| # | Section | Minutes |
|---|---|---|
| 1 | Recap and why this matters | 5 |
| 2 | Reshaping and stacking | 20 |
| 3 | Broadcasting | 25 |
| 4 | Vector and matrix operations (dot, `@`, outer, `einsum`, norm) | 25 |
| 5 | Solving a real naval statics problem with `linalg.solve` | 30 |
| 6 | Eigenvalues/eigenvectors: what PCA does under the hood | 20 |
| 7 | Summary, homework, next class | 15 |

As always: approximate guidance, not a script.


---

## 1. Why this matters


`NB07`–`NB20` all lean on operations *between* arrays, not just within one: combining a per-sensor calibration offset with a full sensor-reading matrix, computing a weighted sum of features, or solving a real physical equilibrium problem. `This class makes those operations explicit and correct, instead of trusting they "just work" inside a library call`.


---

## 2. Reshaping and stacking


`reshape` `changes an array's shape without changing its data or (usually) its memory` — it's a view whenever possible, following `NB03`'s Section 5 rule. `hstack`/`vstack`/`concatenate` combine separate arrays into one, in different directions.


In [ ]:
import numpy as np

readings_flat = np.arange(12.0)   # imagine 12 real sensor readings, logged one after another
print("flat:", readings_flat)

readings_grid = readings_flat.reshape(3, 4)   # 3 sensors x 4 timestamps
print("\nreshaped to (3 sensors, 4 timestamps):\n", readings_grid)

print("\nShares memory with the original?", np.shares_memory(readings_flat, readings_grid))


Now two separate real sensor sequences, combined two different ways.


In [ ]:
sensor_a = np.array([12.1, 12.4, 12.3])
sensor_b = np.array([15.0, 15.2, 14.9])

stacked_rows = np.vstack([sensor_a, sensor_b])       # stacked as two rows
stacked_cols = np.hstack([sensor_a, sensor_b])       # concatenated end to end

print("vstack (2 sensors x 3 readings):\n", stacked_rows)
print("\nhstack (one long sequence of 6 readings):", stacked_cols)


**Try it yourself**: stack `sensor_a`, `sensor_b`, and a third sequence `sensor_c = np.array([11.8, 12.0, 11.9])` into one `(3, 3)` array with `np.vstack`, then use boolean masking to find every individual reading above `13.0` across all three sensors at once (hint: a plain comparison on the whole 2-D array works, since NumPy broadcasts a scalar against any shape).

In [ ]:
sensor_c = np.array([11.8, 12.0, 11.9])
all_sensors = np.vstack([sensor_a, sensor_b, sensor_c])

print("Stacked shape:", all_sensors.shape)
print("Readings above 13.0:", all_sensors[all_sensors > 13.0])


> **Further reading**: [`numpy.reshape` documentation](https://numpy.org/doc/stable/reference/generated/numpy.reshape.html) | [NumPy array manipulation routines](https://numpy.org/doc/stable/reference/routines.array-manipulation.html)


---

## 3. Broadcasting


**Broadcasting** is the rule NumPy uses to combine arrays of *different* shapes without an explicit loop, by mentally "stretching" the smaller array across the larger one — `as long as their shapes are compatible from the trailing dimension inward` (equal, or one of them is `1`). This is exactly how `NB07`'s feature scaling and `NB18`'s grid arithmetic avoid writing a Python `for` loop over every row.

A real example: three sensors, each with its own fixed calibration offset, logged across 4 timestamps. Subtracting a (3,) array of offsets from a (3, 4) array of readings broadcasts the offset across every timestamp automatically.


In [ ]:
readings = np.array([
    [20.1, 20.3, 20.0, 20.4],   # sensor 0, 4 timestamps
    [19.8, 19.9, 20.1, 19.7],   # sensor 1
    [21.0, 21.2, 20.9, 21.1],   # sensor 2
])
calibration_offset = np.array([0.1, -0.2, 0.05])   # one offset per sensor

corrected = readings - calibration_offset[:, np.newaxis]   # shape (3,) -> (3, 1) to broadcast against (3, 4)

print("Raw readings:\n", readings)
print("\nCorrected readings:\n", corrected)


`calibration_offset[:, np.newaxis]` reshapes the offset from `(3,)` to `(3, 1)` — without it, NumPy would try to broadcast `(3,)` directly against `(3, 4)`'s *last* dimension (4), which doesn't match 3, and raise a real `ValueError`. The next cell shows that error on purpose, since recognizing it is a genuinely useful skill.


In [ ]:
try:
    broken = readings - calibration_offset   # (3, 4) vs (3,) -- broadcasts against the WRONG axis
except ValueError as e:
    print(f"ValueError (expected): {e}")


**Try it yourself**: broadcasting isn't limited to matching one array's shape to another's — two 1-D arrays with different *orientations* can combine into a full 2-D grid with no loop and no explicit stacking. Predict the shape, then check: what does `np.array([10, 20, 30])[:, np.newaxis] + np.array([1, 2, 3, 4])` produce?

In [ ]:
row_offsets = np.array([10, 20, 30])[:, np.newaxis]   # shape (3, 1)
col_offsets = np.array([1, 2, 3, 4])                  # shape (4,)
addition_table = row_offsets + col_offsets            # broadcasts to (3, 4)

print("Shape:", addition_table.shape)
print(addition_table)


> **Further reading**: [NumPy broadcasting documentation](https://numpy.org/doc/stable/user/basics.broadcasting.html)


---

## 4. Vector and matrix operations


Three real operations, each answering a different question:
- **Dot product** (`np.dot`, or `@` for matrices): a single number summarizing how two vectors align — the basis of every weighted sum, including a neural network's `weights @ inputs` from `NB11` onward.
- **Outer product** (`np.outer`): every pairwise product between two vectors' elements, producing a full matrix.
- **`einsum`**: a compact, explicit notation for exactly which indices get summed — most useful once an operation is more complex than a plain dot product or matrix multiply.

(Older NumPy code sometimes uses a dedicated `np.matrix` class for 2-D linear algebra — NumPy's own documentation now discourages it in favor of plain `ndarray` with the `@` operator, which is what this class uses throughout.)


In [ ]:
weights = np.array([0.5, 0.3, 0.2])
features = np.array([10.0, 20.0, 5.0])

weighted_sum = np.dot(weights, features)   # a single number: 0.5*10 + 0.3*20 + 0.2*5
print("Dot product (weighted sum):", weighted_sum)

A = np.array([[1.0, 2.0], [3.0, 4.0]])
B = np.array([[5.0, 6.0], [7.0, 8.0]])
print("\nMatrix product A @ B:\n", A @ B)

outer = np.outer(weights, features)
print("\nOuter product (every pairwise product):\n", outer)

einsum_dot = np.einsum("i,i->", weights, features)   # same as np.dot(weights, features)
print("\neinsum reproduces the same dot product:", einsum_dot)


One more real, frequently needed quantity: a vector's **norm** (its magnitude/length) — `np.linalg.norm`. For the mooring-line problem in Section 5, this is exactly how you'd check whether a solved tension is unreasonably large for a real line's rated strength.

In [ ]:
force_vector = np.array([30.0, -40.0])   # a 2-D force, in kN, along x and y
magnitude = np.linalg.norm(force_vector)
print(f"Force components: {force_vector} kN")
print(f"Magnitude (norm): {magnitude:.2f} kN")

# Consistent with the classic 3-4-5 triangle, scaled by 10:
print("Matches sqrt(30^2 + 40^2)?", np.isclose(magnitude, np.sqrt(30.0**2 + 40.0**2)))


> **Further reading**: [`numpy.linalg.norm` documentation](https://numpy.org/doc/stable/reference/generated/numpy.linalg.norm.html) | [Norm (mathematics) (Wikipedia)](https://en.wikipedia.org/wiki/Norm_%28mathematics%29)

> **Further reading**: [Dot product (Wikipedia)](https://en.wikipedia.org/wiki/Dot_product) | [`numpy.einsum` documentation](https://numpy.org/doc/stable/reference/generated/numpy.einsum.html)


---

## 5. Solving a real naval statics problem with `linalg.solve`


A vessel moored alongside a quay is held in place by two mooring lines at fixed angles, resisting a combined wind-and-current force. `For the vessel to stay in static equilibrium, the sum of every force's x- and y-components must be exactly zero`. That gives **two equations in two unknowns** — the two line tensions `T1` and `T2` — a real system of linear equations, exactly the form `numpy.linalg.solve` is built for: `A @ T = b`.

Setup: line 1 pulls at 30° from the vessel's centerline, line 2 at 150° (roughly opposing directions, a realistic bow-and-stern spring configuration), and the environmental force is 40 kN at 200° (pushing the vessel off the quay).


In [ ]:
import numpy as np

angle1_deg, angle2_deg = 30.0, 150.0
env_force_kN, env_angle_deg = 40.0, 200.0

a1, a2, ae = np.radians([angle1_deg, angle2_deg, env_angle_deg])

# Equilibrium: T1 * (cos a1, sin a1) + T2 * (cos a2, sin a2) + F * (cos ae, sin ae) = 0
A = np.array([
    [np.cos(a1), np.cos(a2)],
    [np.sin(a1), np.sin(a2)],
])
b = np.array([
    -env_force_kN * np.cos(ae),
    -env_force_kN * np.sin(ae),
])

T1, T2 = np.linalg.solve(A, b)
print(f"Line 1 tension: {T1:.2f} kN")
print(f"Line 2 tension: {T2:.2f} kN")


**Always verify a solved system against the original equations** — `linalg.solve` will return *a* number even if the setup itself were wrong, so checking the residual is real engineering discipline, not optional.


In [ ]:
residual = A @ np.array([T1, T2]) - b
print("Residual (should be ~0):", residual)
print("Max absolute residual:", np.abs(residual).max())

det_A = np.linalg.det(A)
print(f"\ndet(A) = {det_A:.4f} -- far from zero, so this system has exactly one solution")
print("(a near-zero determinant would mean the two lines pull in almost the same direction,")
print(" making the system poorly conditioned -- physically, an unsafe mooring arrangement)")


**Reading the actual result honestly**: solving this system gives a *negative* tension for one of the two lines. A real mooring line is a rope — it can only pull, never push, so a negative "tension" is not physically achievable; that line would simply go slack, and the vessel would need to rely on fenders or a different line geometry to resist the remaining force. This is not a mistake in the math: it is `linalg.solve` correctly reporting that *this exact* two-line, two-unknown idealization has no physically realizable solution for these particular angles and this environmental force — a real, useful engineering finding, not a bug to explain away.

A real mooring arrangement usually has *more* lines than the 2 unknowns strictly require (redundancy for safety, and exactly the kind of slack-line possibility just seen) — an over-determined system with no single exact solution, solved instead with `numpy.linalg.lstsq` (least squares). That extension is left as this class's homework.

> **Further reading**: [System of linear equations (Wikipedia)](https://en.wikipedia.org/wiki/System_of_linear_equations) | [`numpy.linalg.solve` documentation](https://numpy.org/doc/stable/reference/generated/numpy.linalg.solve.html) | [Mooring (watercraft) (Wikipedia)](https://en.wikipedia.org/wiki/Mooring_%28watercraft%29)


**Try it yourself**: the environmental force's angle (200°) was what produced one negative, physically-impossible tension. Try `env_angle_deg = 250.0` instead (keeping everything else the same) — do both tensions come out positive this time?

In [ ]:
env_angle_deg_2 = 250.0
ae2 = np.radians(env_angle_deg_2)

b2 = np.array([
    -env_force_kN * np.cos(ae2),
    -env_force_kN * np.sin(ae2),
])
T1_b, T2_b = np.linalg.solve(A, b2)
print(f"At 250 degrees -- Line 1 tension: {T1_b:.2f} kN, Line 2 tension: {T2_b:.2f} kN")
print("Both physically realizable (positive)?", T1_b > 0 and T2_b > 0)


---

## 6. Eigenvalues/eigenvectors: what PCA does under the hood


`NB09`'s PCA reduced the real 60-dimensional Sonar dataset down to 2 dimensions using `sklearn.decomposition.PCA`, without showing what happens inside the call. The core operation is: compute the data's **covariance matrix**, then find that matrix's **eigenvectors** (the directions of maximum spread) and **eigenvalues** (how much spread lies along each one). This class computes it manually, on two real, correlated Sonar frequency bands, `and confirms it by hand rather than trusting the library alone`.


In [ ]:
import urllib.request

sonar_url = "https://raw.githubusercontent.com/JuanZapa7a/AINavalEngineering/main/Datasets/sonar.all-data"
raw_lines = urllib.request.urlopen(sonar_url).read().decode("utf-8").strip().split("\n")
sonar_features = np.array([[float(x) for x in line.split(",")[:-1]] for line in raw_lines])

two_bands = sonar_features[:, [10, 11]]   # two real, adjacent frequency bands -- likely correlated
print("Correlation between these two real bands:", np.corrcoef(two_bands.T)[0, 1].round(3))

cov_matrix = np.cov(two_bands.T)
print("\nCovariance matrix:\n", cov_matrix)

eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
print("\nEigenvalues:", eigenvalues)
print("Eigenvectors (columns):\n", eigenvectors)

dominant = eigenvectors[:, np.argmax(eigenvalues)]
print(f"\nDominant direction (largest eigenvalue): {dominant}")
print("This is exactly the direction sklearn's PCA would call 'the first principal component'.")


Seeing the eigenvectors directly on top of the real data makes "direction of maximum spread" concrete instead of abstract:

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(two_bands[:, 0], two_bands[:, 1], s=15, alpha=0.5, color="steelblue")

mean_point = two_bands.mean(axis=0)
for i in range(2):
    # eig() returns complex dtype even for this real, symmetric covariance matrix --
    # .real is safe here since the imaginary part is mathematically guaranteed to be 0
    vec = (eigenvectors[:, i] * np.sqrt(eigenvalues[i]) * 3).real
    ax.annotate("", xy=mean_point + vec, xytext=mean_point,
                arrowprops=dict(arrowstyle="->", color="darkred", linewidth=2))

ax.set_xlabel("Band 10")
ax.set_ylabel("Band 11")
ax.set_title("Real Sonar data with both eigenvector directions overlaid")
ax.axis("equal")
plt.show()


> **Further reading**: [Eigenvalues and eigenvectors (Wikipedia)](https://en.wikipedia.org/wiki/Eigenvalues_and_eigenvectors) | [`numpy.linalg.eig` documentation](https://numpy.org/doc/stable/reference/generated/numpy.linalg.eig.html) | [Covariance matrix (Wikipedia)](https://en.wikipedia.org/wiki/Covariance_matrix)


---

## Class summary

- Reshaping and stacking rearrange data into the shape a computation actually needs, usually without copying (per `NB03`'s view rule).
- Broadcasting combines arrays of compatible-but-different shapes without an explicit loop — genuinely useful, and a real source of `ValueError`s when shapes don't line up as intended.
- `dot`/`@`/outer/`einsum` are different tools for different questions: a single weighted sum, a full matrix product, or every pairwise product.
- A real system of linear equations (mooring-line tensions, in equilibrium) is solved directly with `numpy.linalg.solve`, and checked with the residual and the determinant — solving is not the same as verifying.
- Eigenvectors of a covariance matrix are exactly what PCA (`NB09`) computes under the hood — no longer a black box.

## For the next class (NB05)

A systematic class on Matplotlib: the Figure/Axes object model, styling, subplot layouts, and colormap plots — the plotting foundation every notebook in this course has been using ad hoc since `NB02`.

## Homework / Practice Ideas

1. Extend Section 5's mooring problem to 3 lines (3 unknowns) by adding a third line and a moment-balance equation about the vessel's center of gravity.
2. Make Section 5's system singular on purpose (set both line angles equal) and observe what `np.linalg.det` and `np.linalg.solve` do -- read the actual error message.
3. Research `numpy.linalg.lstsq` and use it to solve an over-determined 3-line mooring system (3 unknowns, but only 2 independent equilibrium equations using this course's 2-D simplification) -- what does "best fit" mean here?
4. Pick two *uncorrelated* Sonar frequency bands (check with `np.corrcoef` first) and repeat Section 6's eigenvector computation -- how does the result differ from the correlated pair used in class?
5. Broadcast a (4,) array of per-timestamp correction factors (not per-sensor) against Section 3's `readings` array -- what shape reshape does `[np.newaxis, :]` need here, compared to the per-sensor case shown in class?

> ***As always: solving a real equation is only half the job -- checking the residual is the other half.***
